# FLEX-445 — Notebook 01: Typical-Section Aeroelastic Foundations

This notebook develops and checks the reduced-order plunge–pitch aeroelastic model used as the foundation for FLEX-445. It compares a Jones/Wagner state-space representation with the classical Theodorsen frequency-domain formulation before the project proceeds to the AGARD 445.6 wing benchmark in Notebook 02.

**Public theory sources used by the project**
- T. Theodorsen, *General Theory of Aerodynamic Instability and the Mechanism of Flutter*, NACA Report 496. NASA NTRS: https://ntrs.nasa.gov/citations/19930090935
- R. T. Jones, *Operational Treatment of the Nonuniform-Lift Theory in Airplane Dynamics*, NACA TN 667. NASA NTRS archive: https://ntrs.nasa.gov/archive/nasa/casi.ntrs.nasa.gov/19930081472.pdf

See the repository `REFERENCES.md`, `MODEL_LIMITATIONS.md`, and `THIRD_PARTY_NOTICES.md` for provenance and claim boundaries.

> Publication copy: code-cell outputs and execution counts are intentionally cleared so GitHub can render the notebook reliably. Re-run the notebook to regenerate results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Educational typical-section model
chord = 1.0                 # m
mass = 20.0                # kg: total modelled section mass

# Radius of gyration about the elastic axis
radius_gyration = 0.25 * chord   # m

# Prescribed uncoupled natural frequencies
f_h_target = 3.0            # Hz
f_alpha_target = 7.0        # Hz

# CG offset from elastic axis
e_cg = 0.0                 # m; initially no inertial coupling

In [ ]:
# Mass moment of inertia about the elastic axis
I_alpha = mass * radius_gyration**2

# Angular natural frequencies, rad/s
omega_h = 2 * np.pi * f_h_target
omega_alpha = 2 * np.pi * f_alpha_target

# Required spring stiffnesses
k_h = mass * omega_h**2             # N/m
k_alpha = I_alpha * omega_alpha**2  # N*m/rad

# Recover natural frequencies
f_h_check = np.sqrt(k_h / mass) / (2 * np.pi)
f_alpha_check = np.sqrt(k_alpha / I_alpha) / (2 * np.pi)

print(f"Pitch inertia: {I_alpha:.4f} kg*m^2")
print(f"Plunge stiffness: {k_h:.2f} N/m")
print(f"Pitch stiffness: {k_alpha:.2f} N*m/rad")
print(f"Recovered plunge frequency: {f_h_check:.3f} Hz")
print(f"Recovered pitch frequency: {f_alpha_check:.3f} Hz")

In [ ]:
from scipy.linalg import eigh

# Structural matrices for coincident CG and elastic axis
M = np.array([
    [mass, 0.0],
    [0.0, I_alpha]
])

K = np.array([
    [k_h, 0.0],
    [0.0, k_alpha]
])

# Solve K @ phi = omega_squared * M @ phi
omega_squared, mode_shapes = eigh(K, M)

if np.any(omega_squared <= 0.0):
    raise ValueError("Non-positive structural eigenvalue detected.")

natural_frequencies = np.sqrt(omega_squared) / (2 * np.pi)

for index, frequency in enumerate(natural_frequencies, start=1):
    print(f"Mode {index}: {frequency:.3f} Hz")

target_frequencies = np.array([f_h_target, f_alpha_target])

assert np.allclose(
    natural_frequencies,
    np.sort(target_frequencies),
    rtol=1e-10,
    atol=1e-12
), "Matrix solution does not match the uncoupled frequencies."

print("Uncoupled frequency check passed.")

In [ ]:
# Assumed CG offset: 10% chord aft of the elastic axis
e_cg = 0.10 * chord

# Preserve the baseline inertia about the CG
I_cg = mass * radius_gyration**2
I_ea = I_cg + mass * e_cg**2

M_coupled = np.array([
    [mass, mass * e_cg],
    [mass * e_cg, I_ea]
])

# Stiffness remains unchanged from the baseline
K_coupled = K.copy()

# Verify positive-definite mass matrix
np.linalg.cholesky(M_coupled)

omega_squared_coupled, modes_coupled = eigh(
    K_coupled, M_coupled
)

if np.any(omega_squared_coupled <= 0.0):
    raise ValueError("Non-positive structural eigenvalue detected.")

frequencies_coupled = (
    np.sqrt(omega_squared_coupled) / (2 * np.pi)
)

print(f"CG offset: {e_cg:.3f} m")
print(f"Inertia about CG: {I_cg:.4f} kg*m^2")
print(f"Inertia about elastic axis: {I_ea:.4f} kg*m^2")

for index, frequency in enumerate(frequencies_coupled):
    baseline = natural_frequencies[index]
    change = 100 * (frequency / baseline - 1)

    print(
        f"Mode {index + 1}: {frequency:.3f} Hz "
        f"({change:+.2f}% from baseline)"
    )

In [ ]:
offset_ratios = np.linspace(0.0, 0.20, 81)
frequency_sweep = np.zeros((len(offset_ratios), 2))

for index, offset_ratio in enumerate(offset_ratios):
    offset = offset_ratio * chord

    M_trial = np.array([
        [mass, mass * offset],
        [mass * offset, I_cg + mass * offset**2]
    ])

    eigenvalues, _ = eigh(K, M_trial)
    frequency_sweep[index] = np.sqrt(eigenvalues) / (2 * np.pi)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(
    100 * offset_ratios, frequency_sweep[:, 0],
    label="Lower-frequency mode"
)
ax.plot(
    100 * offset_ratios, frequency_sweep[:, 1],
    label="Higher-frequency mode"
)

ax.set(
    xlabel="CG offset aft of elastic axis (% chord)",
    ylabel="Natural frequency (Hz)",
    title="Structural frequency sensitivity to CG offset"
)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
from scipy.integrate import solve_ivp

# State order: [h, alpha, h_dot, alpha_dot]
def structural_response(time, state):
    displacement = state[:2]
    velocity = state[2:]

    acceleration = np.linalg.solve(
        M_coupled,
        -K_coupled @ displacement
    )

    return np.concatenate((velocity, acceleration))


# Release from 10 mm plunge with zero pitch and velocity
initial_state = np.array([0.010, 0.0, 0.0, 0.0])
time_points = np.linspace(0.0, 5.0, 2501)

response = solve_ivp(
    structural_response,
    t_span=(time_points[0], time_points[-1]),
    y0=initial_state,
    t_eval=time_points,
    method="DOP853",
    rtol=1e-9,
    atol=1e-11,
    max_step=0.01
)

if not response.success:
    raise RuntimeError(response.message)

displacement = response.y[:2]
velocity = response.y[2:]

kinetic_energy = 0.5 * np.sum(
    velocity * (M_coupled @ velocity), axis=0
)
strain_energy = 0.5 * np.sum(
    displacement * (K_coupled @ displacement), axis=0
)
total_energy = kinetic_energy + strain_energy

relative_energy_error = (
    total_energy - total_energy[0]
) / total_energy[0]

print(f"Initial energy: {total_energy[0]:.6f} J")
print(
    "Maximum relative energy error: "
    f"{np.max(np.abs(relative_energy_error)):.2e}"
)
print(
    "Peak absolute pitch: "
    f"{np.max(np.abs(np.rad2deg(displacement[1]))):.4f} deg"
)

fig, axes = plt.subplots(
    3, 1, figsize=(9, 8), sharex=True
)

axes[0].plot(response.t, displacement[0] * 1000)
axes[0].set_ylabel("Plunge (mm)")
axes[0].set_title("Undamped coupled structural response")

axes[1].plot(
    response.t, np.rad2deg(displacement[1]),
    color="tab:orange"
)
axes[1].set_ylabel("Pitch (deg)")

axes[2].plot(
    response.t, kinetic_energy, label="Kinetic"
)
axes[2].plot(
    response.t, strain_energy, label="Strain"
)
axes[2].plot(
    response.t, total_energy,
    color="black", linestyle="--", label="Total"
)
axes[2].set_ylabel("Energy (J)")
axes[2].set_xlabel("Time (s)")
axes[2].legend()

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Assumed damping ratio for each coupled structural mode
modal_damping = np.array([0.01, 0.01])

omega_sq, phi = eigh(K_coupled, M_coupled)
omega_n = np.sqrt(omega_sq)

C_structural = (
    M_coupled
    @ phi
    @ np.diag(2 * modal_damping * omega_n)
    @ phi.T
    @ M_coupled
)

# State order: [h, alpha, h_dot, alpha_dot]
A_structural = np.block([
    [np.zeros((2, 2)), np.eye(2)],
    [
        -np.linalg.solve(M_coupled, K_coupled),
        -np.linalg.solve(M_coupled, C_structural)
    ]
])

eigenvalues = np.linalg.eigvals(A_structural)
oscillatory_roots = eigenvalues[eigenvalues.imag > 0]
oscillatory_roots = oscillatory_roots[
    np.argsort(oscillatory_roots.imag)
]

for index, root in enumerate(oscillatory_roots, start=1):
    damped_frequency = root.imag / (2 * np.pi)
    damping_ratio = -root.real / abs(root)

    print(
        f"Mode {index}: "
        f"decay rate = {root.real:.4f} 1/s, "
        f"damped frequency = {damped_frequency:.3f} Hz, "
        f"damping ratio = {damping_ratio:.4f}"
    )

print(
    "All structural eigenvalues have negative real parts:",
    bool(np.all(eigenvalues.real < 0))
)

In [ ]:
damped_response = solve_ivp(
    lambda time, state: A_structural @ state,
    t_span=(time_points[0], time_points[-1]),
    y0=initial_state,
    t_eval=time_points,
    method="DOP853",
    rtol=1e-9,
    atol=1e-11,
    max_step=0.01
)

if not damped_response.success:
    raise RuntimeError(damped_response.message)

fig, axes = plt.subplots(
    2, 1, figsize=(9, 6), sharex=True
)

scales = [1000.0, 180.0 / np.pi]
labels = ["Plunge (mm)", "Pitch (deg)"]

for index, ax in enumerate(axes):
    ax.plot(
        response.t,
        response.y[index] * scales[index],
        color="0.65",
        linewidth=1,
        label="Undamped"
    )
    ax.plot(
        damped_response.t,
        damped_response.y[index] * scales[index],
        label="1% modal damping"
    )
    ax.set_ylabel(labels[index])
    ax.grid(True, alpha=0.3)
    ax.legend()

axes[0].set_title("Structural free response")
axes[1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()

In [ ]:
# Assumed aerodynamic parameters for the typical section
rho_air = 1.225             # kg/m^3
section_width = 1.0         # m
section_area = chord * section_width

x_ea = 0.40 * chord         # m from leading edge
x_ac = 0.25 * chord         # m from leading edge
lift_curve_slope = 2 * np.pi  # 1/rad

moment_arm = x_ea - x_ac


def steady_aero_stiffness(speed):
    """Return the matrix mapping [h, alpha] to aerodynamic loads."""
    lift_per_radian = (
        0.5 * rho_air * speed**2
        * section_area * lift_curve_slope
    )

    return np.array([
        [0.0, -lift_per_radian],
        [0.0, moment_arm * lift_per_radian]
    ])


# Check load directions at a small positive pitch angle
test_speed = 30.0
test_displacement = np.array([0.0, np.deg2rad(1.0)])

aero_load = steady_aero_stiffness(test_speed) @ test_displacement

print(f"Plunge force: {aero_load[0]:.3f} N")
print(f"Pitch moment: {aero_load[1]:.3f} N*m")

assert aero_load[0] < 0.0, "Positive pitch should produce upward lift."
assert aero_load[1] > 0.0, "Lift ahead of the EA should produce nose-up moment."

if moment_arm <= 0.0:
    raise ValueError("This divergence formula requires the EA aft of the AC.")

divergence_speed = np.sqrt(
    2 * k_alpha
    / (rho_air * section_area * lift_curve_slope * moment_arm)
)

print(f"Static divergence speed: {divergence_speed:.3f} m/s")

# Verify the effective torsional stiffness at divergence
K_at_divergence = (
    K_coupled - steady_aero_stiffness(divergence_speed)
)

assert abs(K_at_divergence[1, 1]) / k_alpha < 1e-12
print("Static divergence stiffness check passed.")

In [ ]:
speed_values = np.linspace(0.0, 1.1 * divergence_speed, 200)

stiffness_ratio = np.array([
    (K_coupled - steady_aero_stiffness(speed))[1, 1] / k_alpha
    for speed in speed_values
])

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(speed_values, stiffness_ratio)
ax.axhline(0.0, color="black", linewidth=0.8)
ax.axvline(
    divergence_speed,
    color="tab:red",
    linestyle="--",
    label=f"Static divergence: {divergence_speed:.1f} m/s"
)

ax.set(
    xlabel="Airspeed (m/s)",
    ylabel="Effective torsional stiffness / structural stiffness",
    title="Steady aerodynamic stiffness"
)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
semichord = chord / 2
ea_offset = (x_ea - semichord) / semichord
pitch_rate_arm = semichord * (0.5 - ea_offset)

fluid_mass = np.pi * rho_air * section_width * semichord**2

M_added = fluid_mass * np.array([
    [1.0, -ea_offset * semichord],
    [
        -ea_offset * semichord,
        semichord**2 * (0.125 + ea_offset**2)
    ]
])

M_effective = M_coupled + M_added
np.linalg.cholesky(M_effective)


def quasisteady_state_matrix(speed):
    """Return the incompressible typical-section state matrix."""
    if speed < 0:
        raise ValueError("Airspeed must be non-negative.")

    circulation_factor = (
        rho_air * section_width
        * semichord * lift_curve_slope * speed
    )

    load_direction = np.array([-1.0, moment_arm])

    Q_displacement = steady_aero_stiffness(speed)

    Q_velocity = np.outer(
        load_direction,
        circulation_factor * np.array([1.0, pitch_rate_arm])
    )

    # Noncirculatory pitch-rate contributions
    Q_velocity += np.array([
        [0.0, -fluid_mass * speed],
        [0.0, -fluid_mass * speed * pitch_rate_arm]
    ])

    K_effective = K_coupled - Q_displacement
    C_effective = C_structural - Q_velocity

    return np.block([
        [np.zeros((2, 2)), np.eye(2)],
        [
            -np.linalg.solve(M_effective, K_effective),
            -np.linalg.solve(M_effective, C_effective)
        ]
    ])

In [ ]:
from scipy.optimize import brentq


def maximum_growth_rate(speed):
    roots = np.linalg.eigvals(quasisteady_state_matrix(speed))
    return np.max(roots.real)


scan_speeds = np.linspace(0.0, 1.05 * divergence_speed, 601)
growth_rates = np.array([
    maximum_growth_rate(speed) for speed in scan_speeds
])

crossings = np.flatnonzero(
    (growth_rates[:-1] < 0.0) & (growth_rates[1:] >= 0.0)
)

critical_speed = None

if crossings.size:
    index = crossings[0]

    critical_speed = brentq(
        maximum_growth_rate,
        scan_speeds[index],
        scan_speeds[index + 1],
        xtol=1e-10
    )

    critical_roots = np.linalg.eigvals(
        quasisteady_state_matrix(critical_speed)
    )
    critical_root = critical_roots[
        np.argmax(critical_roots.real)
    ]

    critical_frequency = abs(critical_root.imag) / (2 * np.pi)
    reduced_frequency = (
        abs(critical_root.imag) * semichord / critical_speed
    )

    instability_type = (
        "Oscillatory" if critical_frequency > 1e-3 else "Static"
    )

    print(f"First stability crossing: {critical_speed:.3f} m/s")
    print(f"Instability type: {instability_type}")
    print(f"Frequency at crossing: {critical_frequency:.3f} Hz")
    print(f"Reduced frequency: {reduced_frequency:.3f}")
else:
    print("No stable-to-unstable crossing found in the scanned range.")

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(scan_speeds, growth_rates, label="Maximum eigenvalue real part")
ax.axhline(0.0, color="black", linewidth=0.8)

if critical_speed is not None:
    ax.axvline(
        critical_speed, color="tab:orange",
        linestyle="--", label="First stability crossing"
    )

ax.axvline(
    divergence_speed, color="tab:red",
    linestyle=":", label="Static divergence"
)

ax.set(
    xlabel="Airspeed (m/s)",
    ylabel="Maximum growth rate (1/s)",
    title="Quasi-steady aeroelastic stability"
)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Jones approximation to Wagner's function
wagner_weights = np.array([0.165, 0.335])
wagner_rates = np.array([0.0455, 0.3])
instantaneous_fraction = 1.0 - wagner_weights.sum()


def wagner_state_matrix(speed):
    """Return the six-state incompressible aeroelastic matrix."""
    if speed <= 0.0:
        raise ValueError("The wake-state model requires positive airspeed.")

    decay_rates = wagner_rates * speed / semichord

    circulation_factor = (
        rho_air * section_width
        * semichord * lift_curve_slope * speed
    )

    # State order: [h, alpha, h_dot, alpha_dot, lambda_1, lambda_2]
    downwash_map = np.array([
        0.0, speed, 1.0, pitch_rate_arm
    ])

    lift_map = circulation_factor * np.concatenate((
        instantaneous_fraction * downwash_map,
        np.ones(2)
    ))

    force_map = np.outer(
        np.array([-1.0, moment_arm]),
        lift_map
    )

    force_map[:, :2] -= K_coupled
    force_map[:, 2:4] -= C_structural

    # Noncirculatory pitch-rate contributions
    force_map[:, 3] += np.array([
        -fluid_mass * speed,
        -fluid_mass * speed * pitch_rate_arm
    ])

    A = np.zeros((6, 6))
    A[:2, 2:4] = np.eye(2)
    A[2:4, :] = np.linalg.solve(M_effective, force_map)

    A[4:, :4] = np.outer(
        wagner_weights * decay_rates,
        downwash_map
    )
    A[4:, 4:] = -np.diag(decay_rates)

    return A

In [ ]:
A_check = wagner_state_matrix(30.0)

# Eliminate equilibrium wake states from the acceleration equations
static_acceleration_map = (
    A_check[2:4, :2]
    - A_check[2:4, 4:]
    @ np.linalg.solve(A_check[4:, 4:], A_check[4:, :2])
)

expected_static_map = -np.linalg.solve(
    M_effective,
    K_coupled - steady_aero_stiffness(30.0)
)

assert np.allclose(
    static_acceleration_map,
    expected_static_map,
    rtol=1e-10,
    atol=1e-10
)

print("Steady-limit check passed.")

In [ ]:
def wagner_growth_rate(speed):
    roots = np.linalg.eigvals(wagner_state_matrix(speed))
    return np.max(roots.real)


# Avoid zero speed, where wake states have zero decay rates
comparison_speeds = np.linspace(
    0.1, 1.05 * divergence_speed, 1001
)

quasisteady_growth = np.array([
    maximum_growth_rate(speed) for speed in comparison_speeds
])
wagner_growth = np.array([
    wagner_growth_rate(speed) for speed in comparison_speeds
])

crossings = np.flatnonzero(
    (wagner_growth[:-1] < 0.0)
    & (wagner_growth[1:] >= 0.0)
)

wagner_critical_speed = None

if crossings.size:
    index = crossings[0]

    wagner_critical_speed = brentq(
        wagner_growth_rate,
        comparison_speeds[index],
        comparison_speeds[index + 1],
        xtol=1e-10
    )

    roots = np.linalg.eigvals(
        wagner_state_matrix(wagner_critical_speed)
    )
    critical_root = roots[np.argmax(roots.real)]

    frequency = abs(critical_root.imag) / (2 * np.pi)
    reduced_frequency = (
        abs(critical_root.imag) * semichord / wagner_critical_speed
    )

    print(f"Wagner stability crossing: {wagner_critical_speed:.3f} m/s")
    print(f"Frequency at crossing: {frequency:.3f} Hz")
    print(f"Reduced frequency: {reduced_frequency:.3f}")
    print(
        "Instability type:",
        "Oscillatory" if frequency > 1e-3 else "Static"
    )
else:
    print("No stable-to-unstable crossing found in the scanned range.")

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(comparison_speeds, quasisteady_growth, label="Quasi-steady")
ax.plot(comparison_speeds, wagner_growth, label="Wagner approximation")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.axvline(
    divergence_speed, color="tab:red",
    linestyle=":", label="Static divergence"
)

ax.set(
    xlabel="Airspeed (m/s)",
    ylabel="Maximum growth rate (1/s)",
    title="Effect of wake lag on predicted stability"
)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
from scipy.special import hankel2
from scipy.optimize import root


def circulation_response(k, model):
    """Return circulatory response for the exp(i*omega*t) convention."""
    if model == "jones":
        return instantaneous_fraction + np.sum(
            wagner_weights * wagner_rates
            / (wagner_rates + 1j * k)
        )

    if model == "theodorsen":
        h1 = hankel2(1, k)
        h0 = hankel2(0, k)
        return h1 / (h1 + 1j * h0)

    raise ValueError(f"Unknown aerodynamic model: {model}")


def dynamic_stiffness(speed, omega, model):
    """Return the complex dynamic stiffness for harmonic motion."""
    k = omega * semichord / speed
    circulation = circulation_response(k, model)

    downwash_map = np.array([
        1j * omega,
        speed + 1j * omega * pitch_rate_arm
    ])

    circulatory_load = (
        rho_air * section_width * semichord
        * lift_curve_slope * speed * circulation
        * np.outer([-1.0, moment_arm], downwash_map)
    )

    noncirculatory_velocity = np.array([
        [0.0, -fluid_mass * speed],
        [0.0, -fluid_mass * speed * pitch_rate_arm]
    ])

    return (
        K_coupled
        - omega**2 * M_effective
        + 1j * omega * (C_structural - noncirculatory_velocity)
        - circulatory_load
    )


def solve_neutral_point(model, speed_guess, omega_guess):
    """Find a local neutral oscillation near the supplied guess."""
    def residual(log_parameters):
        speed, omega = np.exp(log_parameters)
        matrix = dynamic_stiffness(speed, omega, model)

        # Normalize the determinant using structural stiffness
        determinant = np.linalg.det(
            np.linalg.solve(K_coupled, matrix)
        )
        return np.array([determinant.real, determinant.imag])

    solution = root(
        residual,
        np.log([speed_guess, omega_guess]),
        tol=1e-10
    )

    residual_norm = np.linalg.norm(residual(solution.x))

    if not solution.success or residual_norm > 1e-8:
        raise RuntimeError(
            f"{model} solve failed: {solution.message}; "
            f"residual={residual_norm:.3e}"
        )

    speed, omega = np.exp(solution.x)
    return speed, omega, residual_norm

In [ ]:
if wagner_critical_speed is None:
    raise RuntimeError("A Wagner crossing is needed for the initial guess.")

wagner_roots = np.linalg.eigvals(
    wagner_state_matrix(wagner_critical_speed)
)
neutral_root = wagner_roots[np.argmax(wagner_roots.real)]
omega_guess = abs(neutral_root.imag)

comparison_results = {}

for model in ("jones", "theodorsen"):
    speed, omega, residual_norm = solve_neutral_point(
        model,
        wagner_critical_speed,
        omega_guess
    )

    comparison_results[model] = (speed, omega)

    print(
        f"{model.capitalize()}: "
        f"speed = {speed:.3f} m/s, "
        f"frequency = {omega / (2 * np.pi):.3f} Hz, "
        f"residual = {residual_norm:.2e}"
    )

jones_speed, jones_omega = comparison_results["jones"]
theodorsen_speed, theodorsen_omega = comparison_results["theodorsen"]

assert np.isclose(
    jones_speed, wagner_critical_speed, rtol=1e-7
), "Frequency-domain and state-space speeds disagree."

assert np.isclose(
    jones_omega, omega_guess, rtol=1e-7
), "Frequency-domain and state-space frequencies disagree."

speed_difference = (
    100 * (jones_speed - theodorsen_speed) / theodorsen_speed
)
frequency_difference = (
    100 * (jones_omega - theodorsen_omega) / theodorsen_omega
)

print("State-space / frequency-domain check passed.")
print(f"Jones speed difference vs Theodorsen: {speed_difference:+.3f}%")
print(f"Jones frequency difference vs Theodorsen: {frequency_difference:+.3f}%")

In [ ]:
# Small plunge disturbance with initially unperturbed wake states
initial_aeroelastic_state = np.array([
    0.001, 0.0, 0.0, 0.0, 0.0, 0.0
])

response_times = np.linspace(0.0, 5.0, 2501)
speed_factors = [0.95, 1.05]

fig, axes = plt.subplots(
    2, 2, figsize=(11, 6), sharex=True, sharey="row"
)

for column, factor in enumerate(speed_factors):
    speed = factor * wagner_critical_speed
    A = wagner_state_matrix(speed)

    roots = np.linalg.eigvals(A)
    dominant_root = roots[np.argmax(roots.real)]
    growth_rate = dominant_root.real

    solution = solve_ivp(
        lambda time, state: A @ state,
        t_span=(response_times[0], response_times[-1]),
        y0=initial_aeroelastic_state,
        t_eval=response_times,
        method="DOP853",
        rtol=1e-9,
        atol=1e-11,
        max_step=0.01
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    status = "Stable" if growth_rate < 0.0 else "Unstable"

    print(
        f"U = {speed:.3f} m/s: {status}, "
        f"dominant growth rate = {growth_rate:+.4f} 1/s"
    )

    axes[0, column].plot(
        solution.t, solution.y[0] * 1000
    )
    axes[1, column].plot(
        solution.t, np.rad2deg(solution.y[1]),
        color="tab:orange"
    )

    axes[0, column].set_title(
        f"{factor:.2f} × predicted flutter speed\n"
        f"U = {speed:.2f} m/s — {status}"
    )
    axes[1, column].set_xlabel("Time (s)")

axes[0, 0].set_ylabel("Plunge (mm)")
axes[1, 0].set_ylabel("Pitch (deg)")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
initial_aeroelastic_state = np.array([
    0.001, 0.0, 0.0, 0.0, 0.0, 0.0
])

# Analysis cutoffs, not structural failure limits
plunge_limit = 0.02 * chord
pitch_limit = np.deg2rad(2.0)


def motion_limit(time, state):
    plunge_fraction = abs(state[0]) / plunge_limit
    pitch_fraction = abs(state[1]) / pitch_limit

    return 1.0 - max(plunge_fraction, pitch_fraction)


motion_limit.terminal = True
motion_limit.direction = -1

fig, axes = plt.subplots(
    2, 2, figsize=(11, 6), sharex="col"
)

for column, factor in enumerate([0.95, 1.05]):
    speed = factor * wagner_critical_speed
    A = wagner_state_matrix(speed)

    growth_rate = np.max(np.linalg.eigvals(A).real)
    status = "Stable" if growth_rate < 0.0 else "Unstable"

    solution = solve_ivp(
        lambda time, state: A @ state,
        t_span=(0.0, 5.0),
        y0=initial_aeroelastic_state,
        events=motion_limit,
        dense_output=True,
        method="DOP853",
        rtol=1e-9,
        atol=1e-11,
        max_step=0.01
    )

    if not solution.success:
        raise RuntimeError(solution.message)

    plot_time = np.linspace(0.0, solution.t[-1], 1500)
    states = solution.sol(plot_time)

    axes[0, column].plot(plot_time, states[0] * 1000)
    axes[1, column].plot(
        plot_time, np.rad2deg(states[1]),
        color="tab:orange"
    )

    axes[0, column].set_title(
        f"U = {speed:.2f} m/s — {status}"
    )
    axes[0, column].set_ylabel("Plunge (mm)")
    axes[1, column].set_ylabel("Pitch (deg)")
    axes[1, column].set_xlabel("Time (s)")

    reason = (
        "motion cutoff reached"
        if solution.t_events[0].size
        else "completed"
    )

    print(
        f"U = {speed:.3f} m/s: "
        f"growth rate = {growth_rate:+.4f} 1/s; "
        f"ended at {solution.t[-1]:.3f} s ({reason})"
    )

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

fig.suptitle("Aeroelastic response near the predicted flutter boundary")
fig.tight_layout()
plt.show()